first lets analyse the data 

In [7]:
import os
import glob
import pandas as pd
import numpy as np
from IPython.display import display

# Helper function to format numbers with spaces
def format_num(n):
    if isinstance(n, float):
        return f"{n:,.2f}".replace(",", " ")
    return f"{n:,}".replace(",", " ")

# Path to the raw CSV files
data_dir = '../results/raw/'
csv_files = glob.glob(os.path.join(data_dir, '*.csv'))

total_files = len(csv_files)
total_rows = 0
total_size_bytes = 0
rows_per_file = []
lifespans = []
start_times = []
end_times = []

print(f"Found {total_files} CSV files. Processing...")

for file in csv_files:
    try:
        # Get file size
        total_size_bytes += os.path.getsize(file)
        
        # Read the csv file
        df = pd.read_csv(file)
        num_rows = len(df)
        rows_per_file.append(num_rows)
        total_rows += num_rows
        
        # Try to identify a time column to calculate lifespan and simulation duration
        time_col = None
        for col in df.columns:
            if col.lower() in ['time', 't', 'timestamp', 'simtime']:
                time_col = col
                break
        
        if time_col and num_rows > 0:
            start_t = df[time_col].iloc[0]
            end_t = df[time_col].iloc[-1]
            lifespans.append(end_t - start_t)
            start_times.append(start_t)
            end_times.append(end_t)
    except Exception as e:
        print(f"Error reading {file}: {e}")

# Calculate file sizes
total_size_gb = total_size_bytes / (1024 ** 3)
avg_size_mb = (total_size_bytes / total_files) / (1024 ** 2) if total_files > 0 else 0

# --- Collect Results into a DataFrame ---
results_data = {
    "Metric": [
        "Total CSV files (cars)",
        "Total rows across all files",
        "Total dataset size",
        "Average file size"
    ],
    "Value": [
        f"{format_num(total_files)} files",
        f"{format_num(total_rows)} rows",
        f"{format_num(round(total_size_gb, 2))} GB",
        f"{format_num(round(avg_size_mb, 2))} MB"
    ]
}

if rows_per_file:
    results_data["Metric"].extend([
        "Average rows per file",
        "Highest number of rows",
        "Lowest number of rows"
    ])
    results_data["Value"].extend([
        f"{format_num(int(round(np.mean(rows_per_file))))} rows",
        f"{format_num(np.max(rows_per_file))} rows",
        f"{format_num(np.min(rows_per_file))} rows"
    ])

avg_lifespan_sec = 0
avg_lifespan_steps = 0
if lifespans:
    # The time column in the CSV is already in seconds (e.g., 17002.1)
    avg_lifespan_sec = np.mean(lifespans)
    avg_lifespan_steps = avg_lifespan_sec * 10
    results_data["Metric"].extend([
        "Average lifespan"
    ])
    results_data["Value"].extend([
        f"{format_num(round(avg_lifespan_sec, 2))} seconds ({format_num(int(round(avg_lifespan_steps)))} steps)"
    ])
    
duration_sec = 0
duration_steps = 0
if start_times and end_times:
    sim_start = min(start_times)
    sim_end = max(end_times)
    # The time difference is in seconds
    duration_sec = sim_end - sim_start
    duration_steps = duration_sec * 10
    results_data["Metric"].extend([
        "Simulation start time",
        "Simulation end time",
        "Total simulation duration"
    ])
    results_data["Value"].extend([
        f"{format_num(round(sim_start, 2))} s",
        f"{format_num(round(sim_end, 2))} s",
        f"{format_num(round(duration_sec, 2))} seconds ({format_num(int(round(duration_steps)))} steps)"
    ])

results_df = pd.DataFrame(results_data)

# Display as a pretty HTML dataframe in Jupyter
display(results_df)

# --- Build and Save Markdown Report ---
report_lines = [
    "# Data Analysis Report\n",
    "## General Information",
    f"- **Total CSV files (cars):** {format_num(total_files)} files",
    f"- **Total rows across all files:** {format_num(total_rows)} rows",
    f"- **Total dataset size:** {format_num(round(total_size_gb, 2))} GB",
    f"- **Average file size:** {format_num(round(avg_size_mb, 2))} MB"
]

if rows_per_file:
    report_lines.extend([
        f"- **Average rows per file:** {format_num(int(round(np.mean(rows_per_file))))} rows",
        f"- **Highest number of rows:** {format_num(np.max(rows_per_file))} rows",
        f"- **Lowest number of rows:** {format_num(np.min(rows_per_file))} rows"
    ])

report_lines.extend(["\n## Temporal Information"])

if lifespans:
    report_lines.append(f"- **Average lifespan of a car:** {format_num(round(avg_lifespan_sec, 2))} seconds ({format_num(int(round(avg_lifespan_steps)))} steps)")
    
if start_times and end_times:
    report_lines.append(f"- **Simulation start time:** {format_num(round(sim_start, 2))} s")
    report_lines.append(f"- **Simulation end time:** {format_num(round(sim_end, 2))} s")
    report_lines.append(f"- **Total simulation duration:** {format_num(round(duration_sec, 2))} seconds ({format_num(int(round(duration_steps)))} steps)")
else:
    report_lines.append("- *(Could not find a recognized time column to calculate durations)*")

report_md = "\n".join(report_lines)

report_path = "analysis_report.md"
with open(report_path, "w") as f:
    f.write(report_md)

print(f"\nReport successfully saved to {os.path.abspath(report_path)}")

Found 749 CSV files. Processing...


,Metric,Value
0,Total CSV files (cars),749 files
1,Total rows across all files,12 228 787 rows
2,Total dataset size,2.23 GB
3,Average file size,3.05 MB
4,Average rows per file,16 327 rows
5,Highest number of rows,41 263 rows
6,Lowest number of rows,234 rows
7,Average lifespan,1 632.58 seconds (16 326 steps)
8,Simulation start time,17 002.10 s
9,Simulation end time,22 129.30 s



Report successfully saved to /home/massi/Documents/omnetpp-5.6.2/samples/TrajectoryCollector/custom-scripts/analysis_report.md
